In [1]:
import os, re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

ROOT = "/kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy"
BASE = os.path.join(ROOT, "dr_unified_v2", "dr_unified_v2")
OUT  = "/kaggle/working"

SEED = 42
DR_MIN_GRADE = 1                  # label = 1 if grade >= this. 1 = any DR, 2 = referable DR (grade 2-4)
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15

UNDERSAMPLE_TRAIN = True          # cut No DR in train down to the DR count (per source)
MATCH_RATIO = True                # trim val/test so the final kept data is 70/15/15 again

In [2]:
rows = []
skipped_gf = 0

for split in ["train", "val", "test"]:
    for grade in ["0", "1", "2", "3", "4"]:
        folder = os.path.join(BASE, split, grade)
        for f in os.listdir(folder):
            name = os.path.splitext(f)[0]

            # 1. Skip Gaussian-filtered copies (both naming styles)
            if name.endswith("-GF") or name.startswith("GF-"):
                skipped_gf += 1
                continue

            # 2. Work out the source and the group (patient) ID
            eyepacs = re.fullmatch(r"(\d+)_(left|right)", name)
            if re.fullmatch(r"[0-9a-f]{12}", name):
                source, group_id = "APTOS", f"aptos_{name}"
            elif eyepacs:
                source, group_id = "EyePACS", f"eyepacs_{eyepacs.group(1)}"
            else:
                print("Unexpected file:", split, grade, f)
                continue

            rows.append({
                "filename": f,
                "path": os.path.join(folder, f),
                "source": source,
                "group_id": group_id,
                "grade": int(grade),
                "label": int(int(grade) >= DR_MIN_GRADE),
                "orig_split": split,
            })

df = pd.DataFrame(rows)

print("Skipped GF:", skipped_gf)
print("Rows:", len(df))
print("\nBy source:\n", df["source"].value_counts())
print("\nBy grade:\n", df["grade"].value_counts().sort_index())
print(f"\nDR % (grade >= {DR_MIN_GRADE}):", round(df["label"].mean() * 100, 1))
print("Duplicate filenames:", df["filename"].duplicated().sum())
print("Patients split across the ORIGINAL train/val/test:",
      (df.groupby("group_id")["orig_split"].nunique() > 1).sum())

Skipped GF: 3662
Rows: 88839

By source:
 source
EyePACS    85178
APTOS       3661
Name: count, dtype: int64

By grade:
 grade
0    67148
1     4264
2    14152
3     1066
4     2209
Name: count, dtype: int64

DR % (grade >= 1): 24.4
Duplicate filenames: 0
Patients split across the ORIGINAL train/val/test: 14244


In [3]:
# One row per patient. A patient counts as DR if either eye has DR.
groups = df.groupby("group_id").agg(
    label=("label", "max"),
    source=("source", "first"),
).reset_index()
groups["strata"] = groups["source"] + "_" + groups["label"].astype(str)

train_g, temp_g = train_test_split(
    groups, test_size=1 - TRAIN_FRAC, stratify=groups["strata"], random_state=SEED
)
val_g, test_g = train_test_split(
    temp_g, test_size=TEST_FRAC / (VAL_FRAC + TEST_FRAC), stratify=temp_g["strata"], random_state=SEED
)

split_map = {}
split_map.update({g: "train" for g in train_g["group_id"]})
split_map.update({g: "val" for g in val_g["group_id"]})
split_map.update({g: "test" for g in test_g["group_id"]})
df["split"] = df["group_id"].map(split_map)
df["excluded_reason"] = ""

print("Patients in more than one split:",
      (df.groupby("group_id")["split"].nunique() > 1).sum())
print(df["split"].value_counts())
print((df.groupby("split")["label"].mean() * 100).round(1))

Patients in more than one split: 0
split
train    62195
val      13326
test     13318
Name: count, dtype: int64
split
test     24.4
train    24.4
val      24.7
Name: label, dtype: float64


In [4]:
if UNDERSAMPLE_TRAIN:
    train = df[df["split"] == "train"]
    drop_idx = []
    for src, g in train.groupby("source"):
        pos = g[g["label"] == 1]
        neg = g[g["label"] == 0]
        n_keep = min(len(neg), len(pos))
        keep = neg.sample(n=n_keep, random_state=SEED)
        drop_idx.extend(neg.index.difference(keep.index))
    df.loc[drop_idx, "split"] = "unused"
    df.loc[drop_idx, "excluded_reason"] = "undersampled_no_DR"

print("Train after undersampling:")
t = df[df["split"] == "train"]
print(pd.crosstab(t["source"], t["label"], margins=True))

Train after undersampling:
label        0      1    All
source                      
APTOS     1264   1299   2563
EyePACS  13862  13862  27724
All      15126  15161  30287


In [5]:
def trim_split(df, split_name, target_n, seed):
    part = df[df["split"] == split_name]
    if len(part) <= target_n:
        return
    g = part.groupby("group_id").agg(
        label=("label", "max"), source=("source", "first")
    ).reset_index()
    g["strata"] = g["source"] + "_" + g["label"].astype(str)
    keep_g, _ = train_test_split(
        g, train_size=target_n / len(part), stratify=g["strata"], random_state=seed
    )
    drop = (df["split"] == split_name) & ~df["group_id"].isin(keep_g["group_id"])
    df.loc[drop, "split"] = "unused"
    df.loc[drop, "excluded_reason"] = "trimmed_to_match_ratio"

if MATCH_RATIO:
    n_train = (df["split"] == "train").sum()
    target = int(round(n_train * VAL_FRAC / TRAIN_FRAC))
    trim_split(df, "val", target, SEED)
    trim_split(df, "test", target, SEED)

In [6]:
kept = df[df["split"] != "unused"]

print("Patients in more than one split:",
      (kept.groupby("group_id")["split"].nunique() > 1).sum())

n = kept["split"].value_counts()
print("\nImages per split:\n", n)
print("\nShare of kept data:\n", (n / n.sum() * 100).round(1))
print("\nDR % per split:\n", (kept.groupby("split")["label"].mean() * 100).round(1))
print("\nLabel counts per split:\n", pd.crosstab(kept["split"], kept["label"]))
print("\nSource mix per split:\n", pd.crosstab(kept["split"], kept["source"]))
print("\nGrade mix per split:\n", pd.crosstab(kept["split"], kept["grade"]))
print("\nUnused rows:\n", df.loc[df["split"] == "unused", "excluded_reason"].value_counts())

Patients in more than one split: 0

Images per split:
 split
train    30287
test      6491
val       6483
Name: count, dtype: int64

Share of kept data:
 split
train    70.0
test     15.0
val      15.0
Name: count, dtype: float64

DR % per split:
 split
test     24.4
train    50.1
val      24.5
Name: label, dtype: float64

Label counts per split:
 label      0      1
split              
test    4907   1584
train  15126  15161
val     4892   1591

Source mix per split:
 source  APTOS  EyePACS
split                 
test      267     6224
train    2563    27724
val       268     6215

Grade mix per split:
 grade      0     1     2    3     4
split                              
test    4907   348  1004   76   156
train  15126  2982  9922  713  1544
val     4892   289  1035   99   168

Unused rows:
 excluded_reason
undersampled_no_DR        31908
trimmed_to_match_ratio    13670
Name: count, dtype: int64


In [7]:
df.to_csv(f"{OUT}/DR_all.csv", index=False)
print("Saved", f"{OUT}/DR_all.csv", "rows:", len(df))
print("Use it like: train_df = df[df.split == 'train']")

Saved /kaggle/working/DR_all.csv rows: 88839
Use it like: train_df = df[df.split == 'train']
